# Attention U-Net Medical Segmentation Tutorial

This notebook is the same workflow as notebook 01, using **Kvasir-SEG** for polyp segmentation.

Only the model variant changes here: Attention U-Net uses attention gates on skip connections.


## 1. Install dependencies

Run this once if the packages are not installed:

```powershell
py -m pip install -r requirements.txt
```


In [ ]:
# Optional notebook install. Uncomment and run once if needed.
# %pip install torch numpy pillow matplotlib tqdm requests


## 2. Imports and setup

Set the device. PyTorch will use `cuda` when a supported GPU is available; otherwise it uses CPU.


In [ ]:
from pathlib import Path
import random
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import urllib3
from PIL import Image, ImageOps
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 3. Download Kvasir-SEG

Kvasir-SEG contains endoscopy images and matching masks that mark the polyp region. The download is about 46 MB.


In [ ]:
DATA_URL = "https://datasets.simula.no/downloads/kvasir-seg.zip"

def download_file(url, output_path, verify_ssl=True):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        response = requests.get(url, stream=True, timeout=30, verify=verify_ssl)
    except requests.exceptions.SSLError:
        if not verify_ssl:
            raise
        print("SSL certificate verification failed. Retrying without certificate verification...")
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
        response = requests.get(url, stream=True, timeout=30, verify=False)

    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))

    with output_path.open("wb") as file, tqdm(total=total, unit="B", unit_scale=True) as progress:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)
                progress.update(len(chunk))

def prepare_kvasir(output_dir="data"):
    output_dir = Path(output_dir)
    dataset_dir = output_dir / "Kvasir-SEG"
    images_dir = dataset_dir / "images"
    masks_dir = dataset_dir / "masks"

    if images_dir.exists() and masks_dir.exists():
        print("Dataset already exists:", dataset_dir)
        return dataset_dir

    zip_path = output_dir / "Kvasir-SEG.zip"
    if not zip_path.exists():
        print("Downloading dataset...")
        download_file(DATA_URL, zip_path)

    print("Extracting dataset...")
    output_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(output_dir)

    zip_path.unlink(missing_ok=True)
    return dataset_dir

DATA_DIR = prepare_kvasir("data")
DATA_DIR


## 4. Load images and masks

Each sample returns:

- `image`: tensor with shape `[3, H, W]`
- `mask`: tensor with shape `[1, H, W]`

Masks are converted to binary values: 0 or 1.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

def find_image_mask_pairs(dataset_dir):
    dataset_dir = Path(dataset_dir)
    images_dir = dataset_dir / "images"
    masks_dir = dataset_dir / "masks"

    masks_by_stem = {
        path.stem: path
        for path in masks_dir.iterdir()
        if path.suffix.lower() in IMAGE_EXTENSIONS
    }

    pairs = []
    for image_path in sorted(images_dir.iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        mask_path = masks_by_stem.get(image_path.stem)
        if mask_path is not None:
            pairs.append((image_path, mask_path))

    if not pairs:
        raise RuntimeError("No image/mask pairs found")
    return pairs

def split_pairs(pairs, val_ratio=0.2, seed=42, max_samples=None):
    rng = random.Random(seed)
    pairs = pairs.copy()
    rng.shuffle(pairs)
    if max_samples is not None:
        pairs = pairs[:max_samples]
    val_count = max(1, int(len(pairs) * val_ratio))
    return pairs[val_count:], pairs[:val_count]

class SegmentationDataset(Dataset):
    def __init__(self, pairs, image_size=128, augment=False):
        self.pairs = pairs
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        image = image.resize((self.image_size, self.image_size), Image.BILINEAR)
        mask = mask.resize((self.image_size, self.image_size), Image.NEAREST)

        if self.augment:
            if random.random() < 0.5:
                image = ImageOps.mirror(image)
                mask = ImageOps.mirror(mask)
            if random.random() < 0.2:
                image = ImageOps.flip(image)
                mask = ImageOps.flip(mask)

        image_array = np.asarray(image, dtype=np.float32) / 255.0
        mask_array = (np.asarray(mask, dtype=np.float32) > 127).astype(np.float32)

        image_tensor = torch.from_numpy(image_array).permute(2, 0, 1)
        mask_tensor = torch.from_numpy(mask_array).unsqueeze(0)
        return image_tensor, mask_tensor

pairs = find_image_mask_pairs(DATA_DIR)
len(pairs), pairs[0]

## 5. Preview samples

Check that each image aligns with its mask before training.


In [ ]:
def show_samples(dataset, count=4):
    count = min(count, len(dataset))
    plt.figure(figsize=(count * 3, 6))
    for i in range(count):
        image, mask = dataset[i]
        image_array = image.permute(1, 2, 0).numpy()
        mask_array = mask.squeeze().numpy()

        plt.subplot(2, count, i + 1)
        plt.imshow(image_array)
        plt.title("Image")
        plt.axis("off")

        plt.subplot(2, count, count + i + 1)
        plt.imshow(mask_array, cmap="gray")
        plt.title("Mask")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

sample_dataset = SegmentationDataset(pairs[:4], image_size=128)
show_samples(sample_dataset, count=4)

## 6. Metrics

Pixel accuracy can be misleading in segmentation, so we track:

- **Dice Score**: overlap between predicted and true masks.
- **IoU**: Intersection over Union.
- **Loss**: BCEWithLogitsLoss + Dice Loss.


In [ ]:
def dice_loss(logits, targets, smooth=1.0):
    probs = torch.sigmoid(logits)
    probs = probs.view(probs.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - dice.mean()

def dice_score(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1)
    return ((2.0 * intersection + 1.0) / (union + 1.0)).mean().item()

def iou_score(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection
    return ((intersection + 1.0) / (union + 1.0)).mean().item()

## 7. Build U-Net

U-Net uses an encoder, a decoder, and skip connections. Skip connections pass spatial details from the encoder to the decoder.


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class ResidualDoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.skip = nn.Identity()
        if in_channels != out_channels:
            self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x) + self.skip(x)

class AttentionGate(nn.Module):
    def __init__(self, skip_channels, gate_channels, hidden_channels):
        super().__init__()
        self.skip_proj = nn.Conv2d(skip_channels, hidden_channels, kernel_size=1)
        self.gate_proj = nn.Conv2d(gate_channels, hidden_channels, kernel_size=1)
        self.attention = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, skip, gate):
        gate = F.interpolate(gate, size=skip.shape[2:], mode="bilinear", align_corners=False)
        weights = self.attention(self.skip_proj(skip) + self.gate_proj(gate))
        return skip * weights

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, base_channels=32, residual=False, attention=False):
        super().__init__()
        block = ResidualDoubleConv if residual else DoubleConv
        ch = [base_channels, base_channels * 2, base_channels * 4, base_channels * 8]

        self.down1 = block(in_channels, ch[0])
        self.down2 = block(ch[0], ch[1])
        self.down3 = block(ch[1], ch[2])
        self.bottleneck = block(ch[2], ch[3])
        self.pool = nn.MaxPool2d(2)

        self.up3 = nn.ConvTranspose2d(ch[3], ch[2], kernel_size=2, stride=2)
        self.up_block3 = block(ch[3], ch[2])
        self.up2 = nn.ConvTranspose2d(ch[2], ch[1], kernel_size=2, stride=2)
        self.up_block2 = block(ch[2], ch[1])
        self.up1 = nn.ConvTranspose2d(ch[1], ch[0], kernel_size=2, stride=2)
        self.up_block1 = block(ch[1], ch[0])

        self.use_attention = attention
        if attention:
            self.att3 = AttentionGate(ch[2], ch[2], ch[1])
            self.att2 = AttentionGate(ch[1], ch[1], ch[0])
            self.att1 = AttentionGate(ch[0], ch[0], max(1, ch[0] // 2))

        self.out = nn.Conv2d(ch[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip1 = self.down1(x)
        skip2 = self.down2(self.pool(skip1))
        skip3 = self.down3(self.pool(skip2))
        x = self.bottleneck(self.pool(skip3))

        x = self.up3(x)
        if self.use_attention:
            skip3 = self.att3(skip3, x)
        x = self.up_block3(torch.cat([skip3, x], dim=1))

        x = self.up2(x)
        if self.use_attention:
            skip2 = self.att2(skip2, x)
        x = self.up_block2(torch.cat([skip2, x], dim=1))

        x = self.up1(x)
        if self.use_attention:
            skip1 = self.att1(skip1, x)
        x = self.up_block1(torch.cat([skip1, x], dim=1))

        return self.out(x)


class UNetPlusPlus(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, base_channels=32):
        super().__init__()
        ch = [base_channels, base_channels * 2, base_channels * 4, base_channels * 8]
        self.pool = nn.MaxPool2d(2)

        self.x0_0 = DoubleConv(in_channels, ch[0])
        self.x1_0 = DoubleConv(ch[0], ch[1])
        self.x2_0 = DoubleConv(ch[1], ch[2])
        self.x3_0 = DoubleConv(ch[2], ch[3])

        self.up1_0 = nn.ConvTranspose2d(ch[1], ch[0], kernel_size=2, stride=2)
        self.up2_0 = nn.ConvTranspose2d(ch[2], ch[1], kernel_size=2, stride=2)
        self.up3_0 = nn.ConvTranspose2d(ch[3], ch[2], kernel_size=2, stride=2)
        self.up1_1 = nn.ConvTranspose2d(ch[1], ch[0], kernel_size=2, stride=2)
        self.up2_1 = nn.ConvTranspose2d(ch[2], ch[1], kernel_size=2, stride=2)
        self.up1_2 = nn.ConvTranspose2d(ch[1], ch[0], kernel_size=2, stride=2)

        self.x0_1 = DoubleConv(ch[0] * 2, ch[0])
        self.x1_1 = DoubleConv(ch[1] * 2, ch[1])
        self.x2_1 = DoubleConv(ch[2] * 2, ch[2])
        self.x0_2 = DoubleConv(ch[0] * 3, ch[0])
        self.x1_2 = DoubleConv(ch[1] * 3, ch[1])
        self.x0_3 = DoubleConv(ch[0] * 4, ch[0])
        self.out = nn.Conv2d(ch[0], out_channels, kernel_size=1)

    def forward(self, x):
        x0_0 = self.x0_0(x)
        x1_0 = self.x1_0(self.pool(x0_0))
        x2_0 = self.x2_0(self.pool(x1_0))
        x3_0 = self.x3_0(self.pool(x2_0))

        x0_1 = self.x0_1(torch.cat([x0_0, self.up1_0(x1_0)], dim=1))
        x1_1 = self.x1_1(torch.cat([x1_0, self.up2_0(x2_0)], dim=1))
        x2_1 = self.x2_1(torch.cat([x2_0, self.up3_0(x3_0)], dim=1))
        x0_2 = self.x0_2(torch.cat([x0_0, x0_1, self.up1_1(x1_1)], dim=1))
        x1_2 = self.x1_2(torch.cat([x1_0, x1_1, self.up2_1(x2_1)], dim=1))
        x0_3 = self.x0_3(torch.cat([x0_0, x0_1, x0_2, self.up1_2(x1_2)], dim=1))
        return self.out(x0_3)


class TransUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, base_channels=32, transformer_layers=2, transformer_heads=4):
        super().__init__()
        ch = [base_channels, base_channels * 2, base_channels * 4, base_channels * 8]
        self.pool = nn.MaxPool2d(2)
        self.down1 = DoubleConv(in_channels, ch[0])
        self.down2 = DoubleConv(ch[0], ch[1])
        self.down3 = DoubleConv(ch[1], ch[2])
        self.bottleneck = DoubleConv(ch[2], ch[3])

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=ch[3],
            nhead=transformer_heads,
            dim_feedforward=ch[3] * 2,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)

        self.up3 = nn.ConvTranspose2d(ch[3], ch[2], kernel_size=2, stride=2)
        self.up_block3 = DoubleConv(ch[3], ch[2])
        self.up2 = nn.ConvTranspose2d(ch[2], ch[1], kernel_size=2, stride=2)
        self.up_block2 = DoubleConv(ch[2], ch[1])
        self.up1 = nn.ConvTranspose2d(ch[1], ch[0], kernel_size=2, stride=2)
        self.up_block1 = DoubleConv(ch[1], ch[0])
        self.out = nn.Conv2d(ch[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip1 = self.down1(x)
        skip2 = self.down2(self.pool(skip1))
        skip3 = self.down3(self.pool(skip2))
        x = self.bottleneck(self.pool(skip3))

        batch_size, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        tokens = self.transformer(tokens)
        x = tokens.transpose(1, 2).reshape(batch_size, channels, height, width)

        x = self.up3(x)
        x = self.up_block3(torch.cat([skip3, x], dim=1))
        x = self.up2(x)
        x = self.up_block2(torch.cat([skip2, x], dim=1))
        x = self.up1(x)
        x = self.up_block1(torch.cat([skip1, x], dim=1))
        return self.out(x)


def build_model(name="unet", base_channels=32):
    name = name.lower()
    if name == "unet":
        return UNet(base_channels=base_channels)
    if name == "resunet":
        return UNet(base_channels=base_channels, residual=True)
    if name == "attention_unet":
        return UNet(base_channels=base_channels, attention=True)
    if name in {"unet_plus_plus", "unet++"}:
        return UNetPlusPlus(base_channels=base_channels)
    if name == "transunet":
        return TransUNet(base_channels=base_channels)
    raise ValueError("Choose one of: unet, resunet, attention_unet, unet_plus_plus, transunet")


## 8. Check output shape

The model should map `[batch, 3, height, width]` images to `[batch, 1, height, width]` masks.


In [ ]:
x = torch.randn(2, 3, 64, 64)
for model_name in ["unet", "resunet", "attention_unet"]:
    model = build_model(model_name, base_channels=8)
    y = model(x)
    print(model_name, "=>", tuple(y.shape))

## 9. Training configuration

Start small. `EARLY_STOP_PATIENCE` stops training when validation Dice stops improving.


In [ ]:
MODEL_NAME = "attention_unet"
IMAGE_SIZE = 256
BASE_CHANNELS = 32
MAX_SAMPLES = None
BATCH_SIZE = 4
EPOCHS = 30
LR = 1e-4
EARLY_STOP_PATIENCE = 5
EARLY_STOP_MIN_DELTA = 1e-4

train_pairs, val_pairs = split_pairs(pairs, val_ratio=0.2, seed=42, max_samples=MAX_SAMPLES)
train_dataset = SegmentationDataset(train_pairs, image_size=IMAGE_SIZE, augment=True)
val_dataset = SegmentationDataset(val_pairs, image_size=IMAGE_SIZE, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("train samples:", len(train_dataset))
print("validation samples:", len(val_dataset))


## 10. Training and evaluation functions

The loss combines:

- `BCEWithLogitsLoss` for pixel-level classification.
- `Dice Loss` to improve mask overlap.


In [ ]:
def train_one_epoch(model, loader, optimizer, bce_loss, device):
    model.train()
    running_loss = 0.0
    for images, masks in tqdm(loader, desc="train", leave=False):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = bce_loss(logits, masks) + dice_loss(logits, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

@torch.no_grad()
def validate(model, loader, bce_loss, device):
    model.eval()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    for images, masks in tqdm(loader, desc="valid", leave=False):
        images = images.to(device)
        masks = masks.to(device)
        logits = model(images)
        loss = bce_loss(logits, masks) + dice_loss(logits, masks)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        running_dice += dice_score(logits, masks) * batch_size
        running_iou += iou_score(logits, masks) * batch_size

    total = len(loader.dataset)
    return running_loss / total, running_dice / total, running_iou / total

## 11. Train

The loop saves `best.pt` and stops early when validation Dice does not improve.


In [ ]:
model = build_model(MODEL_NAME, base_channels=BASE_CHANNELS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
bce_loss = nn.BCEWithLogitsLoss()

run_dir = Path("runs") / MODEL_NAME
run_dir.mkdir(parents=True, exist_ok=True)

history = []
best_dice = -1.0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, bce_loss, device)
    val_loss, val_dice, val_iou = validate(model, val_loader, bce_loss, device)

    improved = val_dice > best_dice + EARLY_STOP_MIN_DELTA
    if improved:
        best_dice = val_dice
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_dice": val_dice,
        "val_iou": val_iou,
        "best_val_dice": best_dice,
        "epochs_without_improvement": epochs_without_improvement,
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"dice={val_dice:.4f} | iou={val_iou:.4f} | best_dice={best_dice:.4f}"
    )

    checkpoint = {
        "model_name": MODEL_NAME,
        "base_channels": BASE_CHANNELS,
        "image_size": IMAGE_SIZE,
        "max_samples": MAX_SAMPLES,
        "val_ratio": 0.2,
        "seed": 42,
        "state_dict": model.state_dict(),
    }
    torch.save(checkpoint, run_dir / "last.pt")
    if improved:
        torch.save(checkpoint, run_dir / "best.pt")
        print("Saved new best checkpoint:", run_dir / "best.pt")

    if EARLY_STOP_PATIENCE > 0 and epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(f"Early stopping: no validation Dice improvement for {EARLY_STOP_PATIENCE} epochs.")
        break


## 12. Plot training curves

Training is moving in the right direction when `train_loss` decreases and `val_loss` decreases or stabilizes.


In [ ]:
epochs = [row["epoch"] for row in history]
train_loss_values = [row["train_loss"] for row in history]
val_loss_values = [row["val_loss"] for row in history]

plt.figure(figsize=(7, 4))
plt.plot(epochs, train_loss_values, marker="o", label="train loss")
plt.plot(epochs, val_loss_values, marker="o", label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training curve Attention U-Net")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 13. Preview validation predictions

The red overlay shows the predicted mask on the original image.


In [ ]:
@torch.no_grad()
def show_predictions(model, dataset, count=4, threshold=0.5):
    model.eval()
    count = min(count, len(dataset))
    plt.figure(figsize=(count * 4, 12))

    for i in range(count):
        image, true_mask = dataset[i]
        logits = model(image.unsqueeze(0).to(device))
        prob_mask = torch.sigmoid(logits)[0, 0].cpu().numpy()
        pred_mask = (prob_mask > threshold).astype(np.float32)
        image_array = image.permute(1, 2, 0).numpy()

        plt.subplot(3, count, i + 1)
        plt.imshow(image_array)
        plt.title("Image")
        plt.axis("off")

        plt.subplot(3, count, count + i + 1)
        plt.imshow(true_mask.squeeze().numpy(), cmap="gray")
        plt.title("True mask")
        plt.axis("off")

        plt.subplot(3, count, 2 * count + i + 1)
        plt.imshow(image_array)
        masked_pred = np.ma.masked_where(pred_mask == 0, pred_mask)
        plt.imshow(masked_pred, cmap="Reds", alpha=0.65)
        plt.title("Prediction")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_predictions(model, val_dataset, count=4)

In [ ]:
@torch.no_grad()
def show_predictions_debug(model, dataset, count=4, threshold=0.5):
    model.eval()
    count = min(count, len(dataset))

    # Pick random samples each time
    rng = np.random.default_rng()
    random_indices = rng.choice(len(dataset), size=count, replace=False)

    plt.figure(figsize=(count * 4, 16))

    for col, idx in enumerate(random_indices):
        image, true_mask = dataset[idx]

        logits = model(image.unsqueeze(0).to(device))
        prob_mask = torch.sigmoid(logits)[0, 0].cpu().numpy()
        pred_mask = (prob_mask > threshold).astype(np.float32)

        image_array = image.permute(1, 2, 0).numpy()
        true_mask_array = true_mask.squeeze().numpy()

        print(
            f"Sample index {idx} | "
            f"prob min={prob_mask.min():.4f}, "
            f"max={prob_mask.max():.4f}, "
            f"mean={prob_mask.mean():.4f}, "
            f"true area={true_mask_array.mean():.4f}, "
            f"pred area={pred_mask.mean():.4f}"
        )

        plt.subplot(4, count, col + 1)
        plt.imshow(image_array)
        plt.title(f"Image #{idx}")
        plt.axis("off")

        plt.subplot(4, count, count + col + 1)
        plt.imshow(true_mask_array, cmap="gray")
        plt.title("True Mask")
        plt.axis("off")

        plt.subplot(4, count, 2 * count + col + 1)
        plt.imshow(prob_mask, cmap="gray")
        plt.title("Raw Probability")
        plt.colorbar(fraction=0.046, pad=0.04)
        plt.axis("off")

        plt.subplot(4, count, 3 * count + col + 1)
        plt.imshow(image_array)
        masked_pred = np.ma.masked_where(pred_mask == 0, pred_mask)
        plt.imshow(masked_pred, cmap="Reds", alpha=0.65)
        plt.title(f"Binary Pred > {threshold}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


show_predictions_debug(model, val_dataset, count=10, threshold=0.5)

## 14. Load the best checkpoint later

Run this after defining the model class if you want to reuse the saved checkpoint.


In [ ]:
checkpoint_path = Path("runs") / MODEL_NAME / "best.pt"

if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    loaded_model = build_model(
        checkpoint["model_name"],
        base_channels=checkpoint.get("base_channels", 32),
    ).to(device)
    loaded_model.load_state_dict(checkpoint["state_dict"])
    loaded_model.eval()
    print("Loaded:", checkpoint_path)
else:
    print("Checkpoint not found. Train the model first.")

## 15. Compare model variants

Change `MODEL_NAME` in the configuration cell and train each variant:

- `unet`: simplest baseline.
- `attention_unet`: adds attention gates to skip connections.
- `unet_plus_plus`: adds nested skip connections.
- `transunet`: adds a Transformer block at the bottleneck.

Keep `MAX_SAMPLES`, `IMAGE_SIZE`, and `EPOCHS` fixed for a fair comparison.
